In [ ]:
import os
import glob
import numpy as np
import cv2 as cv
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ntpath
import random
import math

def set_global_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_global_seed(42)

try:
    import torch_directml
    device = torch_directml.device()
    print(f"hardware, using DirectML ({device})")
except:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"hardware, using {device}")

class Config:
    base_directory = '../../../../'
    train_directory = os.path.join(base_directory, 'antrenare')
    valid_directory = os.path.join(base_directory, 'validare')
    output_directory = "output"
    os.makedirs(output_directory, exist_ok=True)

    model_name = "model.pth"
    patch_size = 36
    batch_size = 64
    epochs = 15
    learning_rate = 0.001
    max_mining_images = 300

In [ ]:
def augment_image(patch):
    variations = [patch]
    variations.append(cv.flip(patch, 1))

    height, width = patch.shape
    small = cv.resize(patch, (int(width * 0.7), int(height * 0.7)))
    blurred = cv.resize(small, (width, height))
    variations.append(blurred)

    contrast = np.random.uniform(0.6, 1.4)
    brightness = np.random.randint(-40, 40)
    adjusted = cv.convertScaleAbs(patch, alpha=contrast, beta=brightness)
    variations.append(adjusted)

    return variations

def get_training_data(validation_split=0.15):
    characters = ['fred', 'daphne', 'shaggy', 'velma']
    image_paths = []
    for char in characters:
        image_paths.extend(glob.glob(os.path.join(Config.train_directory, char, '*.jpg')))

    random.Random(42).shuffle(image_paths)

    split_index = int(len(image_paths) * (1 - validation_split))
    train_files = image_paths[:split_index]
    valid_files = image_paths[split_index:]

    def process_set(file_list, is_training=True):
        positives, negatives = [], []

        for path in tqdm(file_list, leave=False, desc="processing data"):
            image = cv.imread(path, cv.IMREAD_GRAYSCALE)
            if image is None: continue

            char_folder = os.path.basename(os.path.dirname(path))
            annotation_file = os.path.join(Config.train_directory, f"{char_folder}_annotations.txt")
            all_annotations = np.loadtxt(annotation_file, dtype=str)
            if all_annotations.ndim == 1:
                all_annotations = np.array([all_annotations])

            image_annotations = all_annotations[all_annotations[:, 0] == ntpath.basename(path)]

            for row in image_annotations:
                x1, y1, x2, y2 = int(row[1]), int(row[2]), int(row[3]), int(row[4])
                face = cv.resize(image[y1:y2, x1:x2], (Config.patch_size, Config.patch_size))

                if is_training:
                    for augmented_patch in augment_image(face):
                        positives.append(augmented_patch / 255.0)
                else:
                    positives.append(face / 255.0)

            img_h, img_w = image.shape
            negative_count = 30 if is_training else 10

            for _ in range(negative_count):
                size = np.random.randint(40, 150)
                if img_w - size <= 0 or img_h - size <= 0: continue
                x = np.random.randint(0, img_w - size)
                y = np.random.randint(0, img_h - size)

                is_valid_negative = True
                for row in image_annotations:
                    face_box = [int(row[1]), int(row[2]), int(row[3]), int(row[4])]
                    ix1, iy1 = max(x, face_box[0]), max(y, face_box[1])
                    ix2, iy2 = min(x + size, face_box[2]), min(y + size, face_box[3])
                    intersection = max(0, ix2 - ix1) * max(0, iy2 - iy1)
                    if intersection > 0:
                        is_valid_negative = False
                        break

                if is_valid_negative:
                    negative_patch = cv.resize(image[y:y+size, x:x+size], (Config.patch_size, Config.patch_size))
                    negatives.append(negative_patch / 255.0)

        return np.array(positives), np.array(negatives)

    train_pos, train_neg = process_set(train_files, is_training=True)
    valid_pos, valid_neg = process_set(valid_files, is_training=False)

    return (train_pos, train_neg), (valid_pos, valid_neg), train_files

(t_pos, t_neg), (v_pos, v_neg), train_file_list = get_training_data()

In [ ]:
class CNNforface1(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Conv2d(128, 256, 4, padding=0), nn.ReLU(), nn.Dropout(0.5),
            nn.Conv2d(256, 2, 1)
        )
        self.to(device)

    def forward(self, x):
        return self.classifier(self.features(x))

def run_training(train_pos, train_neg, valid_pos, valid_neg):
    def get_loader(pos, neg, shuffle=True):
        images = np.concatenate([pos, neg]).reshape(-1, 1, 36, 36)
        labels = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])
        dataset = TensorDataset(torch.tensor(images, dtype=torch.float32), torch.tensor(labels, dtype=torch.long))
        return DataLoader(dataset, batch_size=Config.batch_size, shuffle=shuffle)

    train_loader = get_loader(train_pos, train_neg)
    valid_loader = get_loader(valid_pos, valid_neg, shuffle=False)

    model = CNNforface1()
    optimizer = optim.Adam(model.parameters(), lr=Config.learning_rate)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(Config.epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images).view(images.size(0), 2)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()

        if (epoch + 1) % 5 == 0:
            model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for images, labels in valid_loader:
                    images, labels = images.to(device), labels.to(device)
                    output = model(images).view(images.size(0), 2)
                    preds = torch.argmax(output, dim=1)
                    correct += (preds == labels).sum().item()
                    total += labels.size(0)
            print(f"epoch {epoch+1}, validation accuracy: {correct/total}")
    return model

model_path = os.path.join(Config.output_directory, Config.model_name)

if os.path.exists(model_path):
    trained_model = CNNforface1()
    trained_model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"we have found a model from {Config.model_name}")
else:
    trained_model = run_training(t_pos, t_neg, v_pos, v_neg)
    torch.save(trained_model.state_dict(), model_path)
    print(f"training complete and model saved to {Config.model_name}")

In [ ]:
Config.scale_factor = 0.85
Config.detection_stride = 8
mined_negatives_path = os.path.join(Config.output_directory, "mined_negatives.npy")

def detection(image_path, threshold, model):
    model.eval()
    image_gray = cv.imread(image_path, cv.IMREAD_GRAYSCALE)
    if image_gray is None: return [], []

    current_image = image_gray.copy()
    detections, scores = [], []
    current_scale = 1.0

    while current_image.shape[0] >= Config.patch_size and current_image.shape[1] >= Config.patch_size:
        normalized = current_image.astype(np.float32) / 255.0
        input_tensor = torch.tensor(normalized).unsqueeze(0).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(input_tensor)
            heatmap = output[0, 1, :, :] - output[0, 0, :, :]

        rows, cols = torch.where(heatmap > threshold)
        if len(rows) > 0:
            rows, cols = rows.cpu().numpy(), cols.cpu().numpy()
            confidence_values = heatmap[rows, cols].cpu().numpy()
            inverse_scale = 1.0 / current_scale
            for r, c, score in zip(rows, cols, confidence_values):
                x = int(c * Config.detection_stride * inverse_scale)
                y = int(r * Config.detection_stride * inverse_scale)
                w = int(Config.patch_size * inverse_scale)
                detections.append([x, y, x + w, y + w])
                scores.append(float(score))

        current_image = cv.resize(current_image, None, fx=Config.scale_factor, fy=Config.scale_factor)
        current_scale *= Config.scale_factor

    return detections, scores

def hard_negative_generation(model, train_files, train_pos, train_neg, valid_pos, valid_neg):
    final_model_path = os.path.join(Config.output_directory, Config.model_name)

    if os.path.exists(mined_negatives_path):
        print("loading, we have previously mined hard negatives")
        hard_negatives = np.load(mined_negatives_path)
    else:
        hard_negatives = []
        subset = random.sample(train_files, min(len(train_files), Config.max_mining_images))

        for path in tqdm(subset, desc="we are starting the mining"):
            detections, _ = detection(path, -1.5, model)
            if not detections: continue

            char_folder = os.path.basename(os.path.dirname(path))
            annotation_file = os.path.join(Config.train_directory, f"{char_folder}_annotations.txt")
            all_annotations = np.loadtxt(annotation_file, dtype=str)
            if all_annotations.ndim == 1: 
                all_annotations = np.array([all_annotations])

            image_gt = []
            image_annotations = all_annotations[all_annotations[:, 0] == ntpath.basename(path)]
            for row in image_annotations:
                image_gt.append([int(row[1]), int(row[2]), int(row[3]), int(row[4])])

            image_gray = cv.imread(path, cv.IMREAD_GRAYSCALE)
            for box in detections:
                x1, y1, x2, y2 = box
                is_false_positive = True
                for gx1, gy1, gx2, gy2 in image_gt:
                    newx1, newy1 = max(x1, gx1), max(y1, gy1)
                    newx2, newy2 = min(x2, gx2), min(y2, gy2)
                    if max(0, newx2 - newx1) * max(0, newy2 - newy1) > 0:
                        is_false_positive = False
                        break

                if is_false_positive:
                    if y2 > image_gray.shape[0] or x2 > image_gray.shape[1] or x1 < 0 or y1 < 0: continue
                    crop = cv.resize(image_gray[y1:y2, x1:x2], (Config.patch_size, Config.patch_size))
                    hard_negatives.append(crop / 255.0)

        hard_negatives = np.array(hard_negatives)
        np.save(mined_negatives_path, hard_negatives)
        print(f"mining done, found and saved {len(hard_negatives)} hard negatives.")

    if len(hard_negatives) > 0:
        user_choice = input(f"found {len(hard_negatives)} negatives. do you want to retrain? (y/n): ").lower()
        if user_choice == 'y':
            print("retraining with hard negatives...")
            new_negatives = np.concatenate([train_neg, hard_negatives])
            model = run_training(train_pos, new_negatives, valid_pos, valid_neg)
            torch.save(model.state_dict(), final_model_path)
            print(f"model saved to {final_model_path}")
        else:
            print("skipping retraining as user requested")
    else:
        print("no negatives found, skipping retraining")

    return model

trained_model = hard_negative_generation(trained_model, train_file_list, t_pos, t_neg, v_pos, v_neg)

In [ ]:
test_directory = os.path.join(Config.base_directory, 'evaluare/fake_test')
test_images = sorted(glob.glob(os.path.join(test_directory, '*.jpg')))

def detect_sliding_window(image_path, threshold, model, stride=6):
    model.eval()
    image_gray = cv.imread(image_path, cv.IMREAD_GRAYSCALE)
    if image_gray is None: return [], []

    detections, scores = [], []
    current_image = image_gray.copy()
    current_scale = 1.0

    while current_image.shape[0] >= Config.patch_size and current_image.shape[1] >= Config.patch_size:
        h, w = current_image.shape
        batch_patches, batch_coords = [], []

        for y in range(0, h - Config.patch_size, stride):
            for x in range(0, w - Config.patch_size, stride):
                patch = current_image[y : y + Config.patch_size, x : x + Config.patch_size]
                batch_patches.append(patch.astype(np.float32) / 255.0)
                batch_coords.append((x, y))

                if len(batch_patches) >= 64:
                    tensors = torch.tensor(np.array(batch_patches)).unsqueeze(1).to(device)
                    with torch.no_grad():
                        output = model(tensors)
                        probs = torch.nn.functional.softmax(output, dim=1)
                        face_scores = probs[:, 1].cpu().numpy()

                    for i, score in enumerate(face_scores):
                        if score > threshold:
                            bx, by = batch_coords[i]
                            real_x, real_y = int(bx / current_scale), int(by / current_scale)
                            real_size = int(Config.patch_size / current_scale)
                            detections.append([real_x, real_y, real_x + real_size, real_y + real_size])
                            scores.append(float(score))
                    batch_patches, batch_coords = [], []

        current_image = cv.resize(current_image, None, fx=Config.scale_factor, fy=Config.scale_factor)
        current_scale *= Config.scale_factor

    return detections, scores

def intersection_metrics(box_a, box_b):
    xa, ya = max(box_a[0], box_b[0]), max(box_a[1], box_b[1])
    xb, yb = min(box_a[2], box_b[2]), min(box_a[3], box_b[3])
    intersection = max(0, xb - xa + 1) * max(0, yb - ya + 1)
    area_a = (box_a[2] - box_a[0] + 1) * (box_a[3] - box_a[1] + 1)
    area_b = (box_b[2] - box_b[0] + 1) * (box_b[3] - box_b[1] + 1)
    iou = intersection / float(area_a + area_b - intersection + 1e-6)
    iom = intersection / float(min(area_a, area_b) + 1e-6)
    return iou, iom

def run_nms(boxes, scores, iou_limit=0.10, iom_limit=0.50):
    if len(boxes) == 0: return [], []
    boxes, scores = np.array(boxes), np.array(scores)
    indices = np.argsort(scores)[::-1]
    keep = []
    while len(indices) > 0:
        current = indices[0]
        keep.append(current)
        remove = [0]
        for pos in range(1, len(indices)):
            next_idx = indices[pos]
            iou, iom = intersection_metrics(boxes[current], boxes[next_idx])
            if iou > iou_limit or iom > iom_limit:
                remove.append(pos)
        indices = np.delete(indices, remove)
    return boxes[keep], scores[keep]

print(f"starting final testing on {len(test_images)} images from {test_directory}")

best_threshold = 0.5
final_detections, final_scores, final_filenames = [], [], []

for path in tqdm(test_images, desc="test inference"):
    raw_d, raw_s = detect_sliding_window(path, best_threshold, trained_model)
    nms_d, nms_s = run_nms(raw_d, raw_s)

    final_detections.extend(nms_d)
    final_scores.extend(nms_s)
    final_filenames.extend([ntpath.basename(path)] * len(nms_s))

np.save(os.path.join(Config.output_directory, 'detections_all_faces.npy'), np.array(final_detections))
np.save(os.path.join(Config.output_directory, 'scores_all_faces.npy'), np.array(final_scores))
np.save(os.path.join(Config.output_directory, 'file_names_all_faces.npy'), np.array(final_filenames))

print(f"test inference complete. saved {len(final_detections)} detections.")